# Module 5: Large Language Models (LLMs) — Foundations

Welcome to the start of Module 5. In the previous modules, we went from classical ML (Module 1-2), through Deep Learning & CNNs (Module 3), and into the Transformer architecture (Module 4). Now we take the **biggest leap**: scaling Transformers up to create **Large Language Models**.

This notebook is your conceptual roadmap. We'll explore what makes LLMs tick, and actually *see* the concepts in code.

## 1. What Exactly is an LLM?

At its core, an LLM is just a **next-token predictor on steroids**. You give it some text and it says: *"What's the most likely next word (or piece of a word)?"*

That sounds simple, and it is — conceptually. The magic comes from **scale**. When you train this "next word" game on trillions of words from books, code, Wikipedia, and the entire internet, the model doesn't just learn grammar. It learns facts, reasoning patterns, coding logic, and even a bit of common sense.

### The Next-Token Prediction Loop

```
Input:  "The cat sat on the"
            ↓
      ┌─────────────┐
      │  LLM Engine  │
      │ (Transformer) │
      └─────┬───────┘
            ↓
    Probability Distribution:
      "mat"   → 85%
      "floor" → 10%
      "table" →  4%
      "..."   →  1%
            ↓
    Pick one token → "mat"
            ↓
    New input: "The cat sat on the mat"
    (Repeat until done)
```

**Key insight:** The model doesn't "understand" language like we do. It's incredibly good at *pattern matching* — and when the patterns are learned from enough data, the result looks a lot like understanding.

## 2. The Evolution: How We Got Here

LLMs didn't appear overnight. Here's the journey:

| Era | Approach | How it Works | Limitation |
| :--- | :--- | :--- | :--- |
| 1990s-2000s | **N-grams** | Count word frequencies in training data | No real "understanding", just statistics |
| 2014-2017 | **RNNs / LSTMs** | Process words one-by-one, maintain hidden state | Slow, forgets long sequences |
| 2017 | **Transformers** | Parallel attention over entire sequence | Can't scale beyond a few hundred million params |
| 2018+ | **LLMs (BERT, GPT)** | Transformers + massive data + massive compute | Expensive to train, can hallucinate |
| 2023+ | **Modern LLMs** | Instruction-tuning, RLHF, tool use, multimodal | Still evolving rapidly |

### The Two Families of LLMs

This is a crucial distinction most beginners miss:

- **Encoder models (like BERT):** Read the *entire* text at once. Great for *understanding* tasks — classification, search, entity extraction. They answer: "What does this text mean?"
  
- **Decoder models (like GPT, Llama):** Generate text *one token at a time*, left to right. Great for *generation* tasks — chatbots, writing, coding. They answer: "What comes next?"

When people say "LLM" today, they usually mean decoder models (the ChatGPT family). But encoder models power most of the search engines and recommendation systems you use daily.

## 3. Tokens: The Actual Input to an LLM

LLMs don't read characters or words — they read **tokens**. A token might be:
- A common word like `"the"` → 1 token
- A long word like `"geospatial"` → might be 2-3 tokens (`"ge"`, `"osp"`, `"atial"`)
- A number like `"2024"` → might be 1-2 tokens
- A special character like `"\n"` → 1 token

**Rule of thumb:** 1,000 tokens ≈ 750 English words.

Why does this matter? Because everything the model does — memory, cost, speed — is measured in tokens, not words. Let's actually see this in action.

In [1]:
!pip install tiktoken -q

In [2]:
import tiktoken

# GPT-4's tokenizer (cl100k_base)
enc = tiktoken.get_encoding("cl100k_base")

text = "Geospatial analysis helps us understand patterns on Earth's surface."
tokens = enc.encode(text)

print(f"Original text: {text}")
print(f"Number of tokens: {len(tokens)}")
print(f"Token IDs: {tokens}")
print()

# Let's see what each token actually represents
print("Token breakdown:")
for token_id in tokens:
    print(f"  {token_id:>6} → '{enc.decode([token_id])}'")

Original text: Geospatial analysis helps us understand patterns on Earth's surface.
Number of tokens: 13
Token IDs: [9688, 437, 33514, 6492, 8779, 603, 3619, 12912, 389, 9420, 596, 7479, 13]

Token breakdown:
    9688 → 'Ge'
     437 → 'os'
   33514 → 'patial'
    6492 → ' analysis'
    8779 → ' helps'
     603 → ' us'
    3619 → ' understand'
   12912 → ' patterns'
     389 → ' on'
    9420 → ' Earth'
     596 → ''s'
    7479 → ' surface'
      13 → '.'


In [3]:
# Let's compare token counts for different types of text
examples = [
    "Hello world",
    "Satellite imagery classification using CNN",
    "The quick brown fox jumps over the lazy dog",
    "def calculate_ndvi(red, nir): return (nir - red) / (nir + red)",
    "مرحبا بالعالم",  # Arabic: "Hello World"
]

print(f"{'Text':<60} {'Words':>6} {'Tokens':>7} {'Ratio':>6}")
print("-" * 85)
for text in examples:
    tokens = enc.encode(text)
    words = len(text.split())
    ratio = len(tokens) / words if words > 0 else 0
    print(f"{text:<60} {words:>6} {len(tokens):>7} {ratio:>6.2f}")

Text                                                          Words  Tokens  Ratio
-------------------------------------------------------------------------------------
Hello world                                                       2       2   1.00
Satellite imagery classification using CNN                        5       6   1.20
The quick brown fox jumps over the lazy dog                       9       9   1.00
def calculate_ndvi(red, nir): return (nir - red) / (nir + red)     11      22   2.00
مرحبا بالعالم                                                     2      10   5.00


**Notice something?** Code and non-English text usually need *more* tokens per word. Arabic text is especially expensive because the tokenizer was trained mostly on English. This is a real-world fairness issue in AI — using GPT costs more money for non-English speakers.

## 4. Context Window: The Model's "Working Memory"

The **context window** is the total number of tokens an LLM can "see" at once. This includes:
- Your system prompt
- The entire conversation history
- The model's own previous answers
- Your current question

| Model | Context Window | ≈ Words |
| :--- | :--- | :--- |
| GPT-3 (original) | 2,048 tokens | ~1,500 words |
| Llama 3 (8B) | 8,192 tokens | ~6,000 words |
| GPT-4 Turbo | 128,000 tokens | ~96,000 words |
| Gemini 1.5 Pro | 1,000,000 tokens | ~750,000 words |

### Why does this matter?

If your conversation exceeds the context window, **the model literally forgets the beginning**. It's like trying to have a conversation where you can only remember the last 5 minutes. This is why:
- Long ChatGPT conversations get "confused" over time
- RAG systems exist (we'll build one later in this module!)
- Engineers obsess over prompt length

## 5. Temperature & Top-P: Controlling "Creativity"

Remember the probability distribution from Section 1? Two key parameters control how the model picks from those probabilities:

### Temperature
- **Low (0.1-0.3):** The model almost always picks the *most likely* token. Outputs are predictable and factual. Great for code, math, data extraction.
- **High (0.7-1.0):** The model gives more "creative" (less likely) tokens a chance. Outputs are varied and surprising. Great for stories, brainstorming.
- **0:** Completely deterministic — always picks the top token.

### Top-P (Nucleus Sampling)
Instead of considering ALL possible tokens, top-p says: *"Only consider tokens whose combined probability adds up to P."*

- `top_p = 0.1` → Only the top few tokens (very focused)
- `top_p = 0.9` → Most of the probability mass (more diverse)

**Pro tip:** In practice, adjust either temperature OR top_p, not both. Most people just use temperature.

In [4]:
import ollama

prompt = "Complete this sentence creatively: The satellite detected something unusual —"

# Low temperature = predictable
print("=" * 60)
print("LOW TEMPERATURE (0.1) — Predictable:")
print("=" * 60)
response = ollama.generate(model='llama3', prompt=prompt, options={'temperature': 0.1, 'num_predict': 50})
print(response['response'])

print()

# High temperature = creative
print("=" * 60)
print("HIGH TEMPERATURE (1.0) — Creative:")
print("=" * 60)
response = ollama.generate(model='llama3', prompt=prompt, options={'temperature': 1.0, 'num_predict': 50})
print(response['response'])

LOW TEMPERATURE (0.1) — Predictable:
The satellite detected something unusual — a faint, pulsing glow emanating from the heart of the Amazon rainforest. As scientists scrambled to analyze the data, they realized that the light was not natural, but rather a deliberate signal, repeating itself in a

HIGH TEMPERATURE (1.0) — Creative:
The satellite detected something unusual — a swirling vortex of iridescent lights that seemed to pulse in rhythm with the very heartbeat of the planet, like a celestial drumbeat warning of an ancient power stirring beneath the surface of the Earth. As the data was


**Try it yourself!** Run the cell above multiple times. The low-temperature response will be nearly identical each time, while the high-temperature one will vary wildly.

## 6. Training vs. Inference: The Two Phases

This is something many beginners confuse, so let's be crystal clear:

### Training (done by companies like Meta, OpenAI, Google)
1. **Pre-training**: Feed trillions of words into the model. It learns to predict the next token. This costs **millions of dollars** and takes weeks on thousands of GPUs. The result is a "base model" that can complete text but isn't great at following instructions.
   
2. **Fine-tuning / Alignment**: Take the base model and teach it to be *helpful, harmless, and honest*. Methods include:
   - **SFT (Supervised Fine-Tuning)**: Show it examples of good instruction → response pairs
   - **RLHF (Reinforcement Learning from Human Feedback)**: Humans rank responses, model learns preferences
   - **DPO (Direct Preference Optimization)**: A simpler alternative to RLHF gaining popularity

### Inference (what WE do)
Take a trained model and **run it** — give it prompts, get responses. This is what Ollama does on your machine. No learning happens during inference (the weights don't change).

```
Training:   DATA ──→ [GPU Farm] ──→ MODEL (one time, very expensive)
Inference:  PROMPT ──→ [MODEL] ──→ RESPONSE (every time you chat, cheap)
```

## 7. Scale & Quantization: Running Giants on Your Laptop

### The Scale Problem
- **ResNet (Module 3)**: ~25 million parameters
- **BERT**: 110 million parameters
- **Llama 3 (8B)**: 8 *billion* parameters
- **GPT-4**: rumored 1.8 *trillion* parameters

An 8B model stored in 16-bit precision = **16 GB of RAM** just for the weights. Most laptops can't handle that.

### The Solution: Quantization
Quantization compresses the model by reducing the precision of each number:

| Precision | Bits per Parameter | 8B Model Size | Quality Loss |
| :--- | :--- | :--- | :--- |
| FP16 (original) | 16 bits | ~16 GB | None |
| Q8 | 8 bits | ~8 GB | Negligible |
| **Q4_K_M** | 4 bits | **~4.7 GB** | Very small |
| Q2 | 2 bits | ~2.5 GB | Noticeable |

**Q4_K_M** is the sweet spot that Ollama uses by default — 4-bit quantization with minimal quality loss.

### Quantization Formats
- **GGUF**: The standard for CPU+GPU inference. Used by Ollama, llama.cpp. Works on any hardware.
- **GPTQ / AWQ**: GPU-only formats. Faster but require a good GPU.
- **EXL2**: Optimized for ExLlamaV2. Fastest GPU inference but least compatible.

In [5]:
# Let's see what models you have and their sizes
import ollama

models = ollama.list()

# Handle both old and new Ollama API formats
model_list = models.get('models', []) if isinstance(models, dict) else getattr(models, 'models', [])

print(f"{'Model Name':<30} {'Size (GB)':>10} {'Parameters':>12}")
print("-" * 55)
for model in model_list:
    if isinstance(model, dict):
        name = model.get('name', 'unknown')
        size_gb = model.get('size', 0) / (1024**3)
        params = model.get('details', {}).get('parameter_size', 'N/A')
    else:
        name = getattr(model, 'model', getattr(model, 'name', 'unknown'))
        size_gb = getattr(model, 'size', 0) / (1024**3)
        details = getattr(model, 'details', None)
        params = getattr(details, 'parameter_size', 'N/A') if details else 'N/A'
    print(f"{name:<30} {size_gb:>9.1f}  {params:>12}")

## 8. Why 'Large' Matters: Emergent Abilities

Here's the most fascinating thing about LLMs: at a certain scale, they develop abilities they were **never explicitly trained for**. This is called **emergence**.

| Ability | Small Models | Large Models (8B+) |
| :--- | :--- | :--- |
| Grammar | ✅ | ✅ |
| Translation | ❌ | ✅ |
| Math reasoning | ❌ | ✅ |
| Code generation | ❌ | ✅ |
| Chain-of-thought | ❌ | ✅ |

Nobody told GPT-4 how to solve math word problems — it just learned to do it by seeing enough examples of mathematical reasoning in its training data. This is both exciting and a bit unsettling.

### The Scaling Laws
Research (notably from Anthropic and OpenAI) shows that model performance improves *predictably* with three things:
1. **More parameters** (bigger model)
2. **More data** (more training text)
3. **More compute** (more training time)

This is why companies are racing to build bigger models — the returns haven't stopped yet.

## 9. The Open-Source LLM Landscape (2024-2025)

One of the most exciting developments: you can now run state-of-the-art models **for free on your own machine**.

| Model Family | Creator | Sizes | Standout Feature |
| :--- | :--- | :--- | :--- |
| **Llama 3.1 / 3.2** | Meta | 1B-405B | Best open-source all-rounder |
| **Mistral / Mixtral** | Mistral AI | 7B-8x22B | Excellent efficiency |
| **Gemma 2** | Google | 2B-27B | Strong for its size |
| **Qwen 2.5** | Alibaba | 0.5B-72B | Excellent multilingual + coding |
| **Phi-3** | Microsoft | 3.8B-14B | Punches way above its weight |
| **DeepSeek-V2/V3** | DeepSeek | 16B-236B | MoE architecture, great at coding |

You can run any of these through Ollama with a single command!

## 10. Your Turn! 🚀

Let's put some of this knowledge to work. Try these experiments:

In [6]:
# EXERCISE 1: Tokenize different types of text and compare
# Try tokenizing: a paragraph of English, a code snippet, and some Urdu/Arabic text
# Which uses the most tokens per word? Why?

your_texts = [
    "Remote sensing satellites orbit Earth collecting multispectral imagery data",
    "def calc_ndvi(nir, red): return (nir - red) / (nir + red)",
    "زمین کی سطح کا تجزیہ سیٹلائٹ سے کیا جاتا ہے",  # Urdu: "Earth surface is analyzed via satellite"
]

for text in your_texts:
    tokens = enc.encode(text)
    print(f"Text: {text[:50]}...")
    print(f"  Words: {len(text.split())}, Tokens: {len(tokens)}, Ratio: {len(tokens)/max(len(text.split()),1):.2f}")
    print()

In [7]:
# EXERCISE 2: Experiment with temperature
# Run this cell 3 times and observe how the output changes

import ollama

prompt = "Name 3 applications of satellite imagery in agriculture."

# Try changing this value: 0, 0.3, 0.7, 1.0, 1.5
TEMPERATURE = 0.7

response = ollama.generate(
    model='llama3', 
    prompt=prompt, 
    options={'temperature': TEMPERATURE}
)
print(f"Temperature: {TEMPERATURE}")
print(response['response'])

Temperature: 0.7
Here are three applications of satellite imagery in agriculture:

1. **Crop Monitoring and Yield Prediction**: Satellites equipped with multispectral or hyperspectral sensors can capture images of crops, allowing farmers to monitor crop health, growth, and development. This information can be used to predict yields, detect early signs of stress or disease, and make informed decisions about irrigation, fertilization, and pest control.
2. **Precision Farming and Field Management**: Satellite imagery can help farmers identify areas within a field that require specific attention, such as adjusting fertilizer application rates or changing planting dates. This precision farming approach can lead to increased yields, reduced waste, and lower environmental impact.
3. **Acreage Mapping and Land Use Analysis**: Satellites like Landsat or Sentinel-2 can provide high-resolution images of large areas, allowing farmers and agricultural organizations to map acreage, track land use ch

---

**Next notebook:** [Local LLM Setup](./local_llm_setup.ipynb) — Let's get Ollama running on your machine and pull your first model.